# Streams processing MODES
- Sagarmatha
- 1.0.0

## Context

   ![Pipeline](image/Spark_Streaming.png)

## Refrences
1. [Python API for Apache Spark](https://spark.apache.org/docs/latest/api/python/);

## Project creation and configuration
- `JAVA_HOME` must be set for PySpark to be imported correctly.
- `spark.sql.shuffle.partitions` sets the default number of partitions Spark creates when it redistributes data. For example during joins, grouping, aggregations, sorting or distinct etc.

In [ ]:
import os
import logging

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pathlib import Path
import shutil
from pyspark.sql.types import *
from pyspark.sql.streaming import DataStreamReader, StreamingQuery
from delta import *
from time import time, sleep
from random import random, randint

import dbutils
from datetime import timedelta, datetime

if "JAVA_HOME" not in os.environ:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

builder = (
    SparkSession
        .builder
        .master("local[*]")
        .appName("Sagarmatha")
        .config("spark.driver.memory", "2g")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

session = configure_spark_with_delta_pip(builder).getOrCreate()
session.conf.set("spark.sql.shuffle.partitions", "10")
session.conf.set("spark.sql.debug.maxToStringFields", "100")
session.conf.set("spark.sql.shuffle.partitions", "4")

DATABASE_NAME = "himalaya"
VOLUME_LOCATION = "spark-volume"
VOLUME_NAME = "tmp"
OPTIMISE_PERIOD = 5

## Datastore creation and configuration
With creation of database `himalaya` the datastore `spark-warehouse` will be automatically created too.

In [ ]:
session.sql(f"DROP DATABASE IF EXISTS {DATABASE_NAME} CASCADE")
session.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
session.sql(f"USE DATABASE {DATABASE_NAME}")

Creating a volume `VOLUMEN_NAME` at the location `VOLUME_LOCATION`.

In [ ]:
shutil.rmtree(Path(VOLUME_LOCATION, VOLUME_NAME), ignore_errors=True)
Path(VOLUME_LOCATION, VOLUME_NAME).mkdir(parents=True, exist_ok=True)

Zdefiniowanie domyślnej lokalizacji punktów kontrolnych zapisu na podstawie lokalizacji bieżącej bazy danych.

In [ ]:
def get_checkpoint_location():
    checkpoint_location = Path(VOLUME_LOCATION, VOLUME_NAME) / "checkpoints" / datetime.now().strftime("%Y%m%d_%H%M%S")
    #shutil.rmtree(Path(VOLUME_LOCATION, VOLUME_NAME), ignore_errors=True)
    checkpoint_location.mkdir(parents=True, exist_ok=True)
    #checkpoint_location = checkpoint_base.mkdir(parents=True, exist_ok=True)
    return checkpoint_location

#print(get_checkpoint_location())
#timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#checkpoint_location = checkpoint_base / "checkpoints" / timestamp
#checkpoint_location.mkdir(parents=True, exist_ok=True)

## Analysis ⟶ Analysis of temporal aspects in data stream processing EVENT TIME vs. INGESTION TIME vs. PROCESSING TIME

   ![Pipeline](image/TIMES.jpg)

In [ ]:
# Utworzenie ramki danych generującej, w formie odczytu, strumień danych (and. data stream) składający się z:
# * pól domyślnych (timestamp, value);
# * pól generowanych (event_time, ingestion_time, processing_time).
events_stream_df = (
    session
        .readStream
        .format("rate")
        .option("rowsPerSecond", 5)
        .option("numPartitions", 1)
        .load()
        .withColumn("event_time", F.col("timestamp"))
        .withColumn("ingestion_time", F.expr("event_time + make_interval(0, 0, 0, 0, 0, 0, (rand()))"))
        .withColumn("processing_time", F.current_timestamp().cast("timestamp"))
)

# Wyświetlenie schematu ramki danych:
# * z polami domyślnymi tj. timestamp, value
# * oraz z polami generowanymi w przetwarzaniu strumienia danych tj. event_time, ingestion_time, processing_time.
events_stream_df.printSchema()

> What ...?

In [ ]:
# Utworzenie zapytania przetwarzającego, w formie zapisującej, strumień danych (ang. streaming query) do pamięci ulotnej.
events_stream_query = (
    events_stream_df
        .writeStream
        .format("memory")
        .queryName("analysis")
        #.trigger(availableNow=True)
        #.trigger(once=True)
        .trigger(processingTime="1 seconds")
        .option("checkpointLocation", str(get_checkpoint_location().resolve()))
        .start()
)

def process_query(query, timeout=20):
    """Process a streaming query."""
    if query is None:
        return
    if not query.isActive:
        return
    if not query.awaitTermination(timeout=timeout):
        try:
            query.stop()
        except Exception as e:
            print("Wykonanie stop() spowodowało:", e)
        query.awaitTermination(timeout=60)
    print("Zapytanie zakończone. Status:", query.status)

# Uruchomienia a następnie zatrzymanie zapytania przetwarzającego, w formie zapisującej, strumień danych do pamięci ulotnej.
# Zatrzymanie następuje domyślnie po 20 s.
process_query(events_stream_query, timeout=20)

# Wyświetlenie danych z uwzględnieniem czasów:
# - EVENT TIME
# - INGESTION TIME
# - PROCESSING TIME
session.sql("SELECT * FROM analysis").show(truncate=False)

### Processing 1 ⟶ Data stream MICRO-BATCH processing with persistent storage into DELTA TABLE

In [ ]:
# Usuwanie tabeli z katalogu.
session.sql("DROP TABLE IF EXISTS weather")

# Tworzenie tabeli Delta Lake z wykorzystaniem API PySpark.
weather = (
    DeltaTable.createOrReplace(session)
        .tableName("weather")
        .addColumn("id", dataType="long", nullable=False)
        .addColumn("datetime", dataType="timestamp", nullable=False)
        .addColumn("air_temperature", dataType="real")
        .addColumn("humidity", dataType="integer")
        .addColumn("peak_id", dataType="integer")
        .partitionedBy("peak_id")
        .execute()
)

# Informacje o tabeli delta weather.
session.sql("DESCRIBE DETAIL weather").show(truncate=False)

> Compare ... ?

In [ ]:
# Zdefiniowanie lokalizacji tabeli w systemie plików HDFS via nazwa tabeli zarejestrowanej w katalogu (HIVE METASTORE, IN-MEMORY, UNITY CATALOG).
weather_location = session.sql("DESCRIBE DETAIL weather").select("location").first()["location"]

# Wyświetlenie lokalizacji.
display(weather_location)
session.sql("DESCRIBE DETAIL weather").show(truncate=False)

In [ ]:
print(session.conf.get("spark.sql.catalogImplementation"))
session.sql("SHOW CATALOGS").show()
session.sql("SHOW NAMESPACES").show()
session.sql("SELECT current_catalog(), current_database()").show()

In [ ]:
from pyspark.sql import DataFrame

# Utworzenie funkcji tworzącej lokalizacje i nazwy punktów kontrolnych zapisu.
# Punkty kontrolne zapisu, w przypadku przetwarzania strumieni, pozwalają zwiększyć bezpieczeństwo danych.
# Zapewniają możliwość odtworzenia strumienia z ostatniego zapisanego punktu kontrolnego.
def get_random_checkpoint(peak_id: LongType):
    return get_checkpoint_location() / str(peak_id) / str(randint(0, 100000))

# Utworzenie funkcji zwracającej strumień danych (ang. data stream) pogodowych generowanych pseudolosowo.
def start_streaming_weather_data(peak_id: LongType) -> DataStreamReader:
    measurements_stream_df = (
        session
            .readStream
            .format("rate")
            .option("rowsPerSecond", 10)
            .load()
            .withColumn("id", F.col("value"))
            .withColumn("datetime", F.col("timestamp"))
            .withColumn("air_temperature", F.round(10 - (F.rand() * 50), 2).cast("real"))
            .withColumn("humidity", (F.rand() * 100).cast("integer"))
            .withColumn("peak_id", F.lit(peak_id))
            .select("id", "datetime", "air_temperature", "humidity", "peak_id")
    )
    return measurements_stream_df

# Tworzenie funkcji zwracającej zapytanie przetwarzające strumień danych (ang. streaming query) do tabeli Delta.
def start_processing_weather_data(peak_id: LongType, measurements_stream_df: DataFrame) -> StreamingQuery:
    measurements_stream_query = (
        measurements_stream_df
            .writeStream
            .format("delta")
            .option("checkpointLocation", str(get_random_checkpoint(peak_id).resolve()))
            .trigger(processingTime="5 seconds")
            #.trigger(continuous="1 second")
            #.trigger(availableNow=True)
            #.trigger(once=True)
            .toTable("weather")
    )
    return measurements_stream_query

# Tworzenie funkcji realizującej zatrzymanie zapytania przetwarzającego strumień danych.
def stop_streaming_weather_data(peak_id: LongType, mesurements_stream_query: StreamingQuery, period: int) -> None:
    """Stop a streaming query."""

    if mesurements_stream_query is None:
        return
    if not mesurements_stream_query.isActive:
        return
    if not mesurements_stream_query.awaitTermination(timeout=period):
        try:
            mesurements_stream_query.stop()
        except Exception as e:
            print("Wykonanie stop() spowodowało:", e)
        mesurements_stream_query.awaitTermination(timeout=60)
    print("Zapytanie przetwarzające strumień danych dla szczytu %s zostało zatrzymane." % peak_id, mesurements_stream_query.status)

    #mesurements_stream_query.stop()
    print("Zapytanie przetwarzające strumień danych dla szczytu %s zostało zatrzymane." % peak_id)
    #dbutils.fs.rm(checkpoint_location + "weather/" + str(peak_id), True)
    #print("Punkty kontrolne zapisu w lokalizacji %s zostały usunięte." % (checkpoint_location + str(peak_id)))

# Tworzenie funkcji realizującej generowanie i przetwarzanie strumienia danych pogodowych dla wskazanego szczytu przez wskazany okres.
def streaming_weather_data(peak_id: LongType, period: int) -> None:
    weather_stream_query = start_processing_weather_data(peak_id, start_streaming_weather_data(peak_id))
    if not weather_stream_query.awaitTermination(timeout=period):
        stop_streaming_weather_data(peak_id, weather_stream_query, period)

In [ ]:
# Generowanie strumienia danych dla szczytu 1 przez okres 20 s.
streaming_weather_data(1, 20)

In [ ]:
# Generowanie strumienia danych dla szczytu 2 przez okres 20 s.
streaming_weather_data(2, 20)

In [ ]:
# Wyświetlanie aktualnej liczby rekordów po rozpoczęciu generowania strumienia danych
session.sql("SELECT peak_id, COUNT(*) FROM weather GROUP BY peak_id ORDER BY 1 ASC").show(truncate=False);

# Wyświetlanie wszystkich danych
session.sql("SELECT * FROM weather ORDER BY 2 ASC").show(truncate=False, n=100);

In [ ]:
# Usuwanie danych z tabeli
weather.delete()

# Wyświetlanie wszystkich danych
session.sql("SELECT * FROM weather ORDER BY 2 ASC").show(truncate=False);

### Processing 2 ⟶ Data stream MICRO-BATCH UDA processing with persistent storage into DELTA TABLE

In [ ]:
logger = logging.getLogger("Batch streaming logger")
logger.setLevel(logging.INFO)

# Tworzenie funkcji wykonującej przetwarzanie (mikro)wsadowe (ang. (micro)batch processing).
def process_batch(batch_output: DataFrame, batch_id: int) -> None:
    """
    Parametery:
         batch_output: ramka danych
         batch_id: numer operacji przetwarzania (mikro)wsadowego
    """
    logger.info(f"Przetwarzanie (mikro)wsadowe nr {batch_id}.")
    print("Przetwarzanie (mikro)wsadowe nr %s." % batch_id)

    weather = DeltaTable.forName(session, "weather")

    weather \
        .alias("target") \
        .merge(
            batch_output.alias("source"),
            "target.datetime = source.datetime" or "target.peak_id = source.peak_id"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

# Tworzenie funkcji zwracającej zapytanie przetwarzające (mikro)wsadowo strumień danych (ang. (micro)batch streaming query).
def start_batch_processing_weather_data(peak_id: LongType, data_stream: DataFrame) -> StreamingQuery:
    streaming_query = (
        data_stream.writeStream
            .format("delta")
            #.trigger(availableNow=True)
            #.trigger(once=True)
            .trigger(processingTime="5 seconds")
            .option("checkpointLocation", get_random_checkpoint(peak_id))
            .foreachBatch(process_batch)
            .start()
    )
    return streaming_query

# Tworzenie funkcji realizującej generowanie strumienia danych pogodowych z przetwarzaniem (mikro)wsadowym dla wskazanego szczytu przez wskazany okres
def batch_streaming_weather_data(peak_id: LongType, period: int) -> None:
    streaming_data = start_streaming_weather_data(peak_id)
    streaming_query = start_batch_processing_weather_data(peak_id, streaming_data)
    sleep(period)
    if streaming_query.isActive:
        stop_streaming_weather_data(peak_id, streaming_query, period)

In [ ]:
# Generowanie strumienia danych pogodowych z przetwarzaniem (mikro)wsadowym dla szczytu 1 przez okres 30 s.
batch_streaming_weather_data(1, 30)

In [ ]:
# Generowanie strumienia danych pogodowych z przetwarzaniem (mikro)wsadowym 2 przez okres 30 s.
batch_streaming_weather_data(2, 30)

In [ ]:
# Wyświetlanie aktualnej liczby rekordów po rozpoczęciu generowania strumienia danych
session.sql("SELECT COUNT(*) FROM weather").show(truncate=False);

# Wyświetlanie wszystkich danych
session.sql("SELECT * FROM weather ORDER BY 2 ASC").show(truncate=False, n=1000);

### Processing 3 ⟶ Data stream MICRO-BATCH UDA processing with OPTIMIZATION and persistent storage into DELTA TABLE

In [ ]:
logger = logging.getLogger("Optimise logger")
logger.setLevel(logging.INFO)

# Tworzenie funkcji optymalizującej tabelę Delta Lake poprzez porządkowanie danych ZORDER.
def optimise_table(table_name: str, column_name: str) -> None:
    """
    Parametery:
         table_name: optymalizowana tabela
         column_name: kolumna / lista kolumn wg których zostanie wyokonana operacja ZORDER BY
    """
    start = time()
    print("Optymalizowanie tabeli %s." % table_name)
    logger.info(f"Optymalizowanie tabeli {table_name}.")
    query = f"OPTIMIZE {table_name} ZORDER BY ({column_name})"
    session.sql(query)
    end = time()
    print("Tabela zoptymalizowana/uporządkowana w czasie %s s." % (end - start))
    logger.info(f"Tabela zoptymalizowana/uporządkowana w czasie {end - start} s.")

In [ ]:
logger = logging.getLogger("Batch streaming logger")
logger.setLevel(logging.INFO)

# Tworzenie funkcji wykonującej przetwarzanie (mikro)wsadowe (ang. batch processing) z optymalizacją
def process_batch(batch_output: DataFrame, batch_id: int) -> None:
    """
    Parametery:
         batch_output: ramka danych
         batch_id: numer operacji przetwarzania (mikro)wsadowego
    """
    logger.info(f"Przetwarzanie (mikro)wsadowe nr {batch_id}.")
    print("Przetwarzanie (mikro)wsadowe nr %s." % batch_id, flush=True)

    weather = DeltaTable.forName(session, "weather")

    weather \
        .alias("target") \
        .merge(
            batch_output.alias("source"),
            "target.datetime = source.datetime" or "target.peak_id = source.peak_id"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

    if batch_id % OPTIMISE_PERIOD == 0:
        optimise_table("weather", "humidity")

In [ ]:
# Generowanie strumienia danych pogodowych z przetwarzaniem (mikro)wsadowym oraz optymalizacją dla szczytu 3 przez okres 50 s.
batch_streaming_weather_data(3, 50)

In [ ]:
# Generowanie strumienia danych pogodowych z przetwarzaniem (mikro)wsadowym oraz optymalizacją dla szczytu 4 przez okres 50 s.
batch_streaming_weather_data(4, 50)

In [ ]:
# Wyświetlanie aktualnej liczby rekordów po rozpoczęciu generowania strumienia danych
session.sql("SELECT COUNT(*) FROM weather").show(truncate=False);

# Wyświetlanie wszystkich danych
session.sql("SELECT * FROM weather ORDER BY 2 ASC").show(truncate=False, n=1000);

### Processing 4 ⟶ Data stream CONTINOUS processing

In [ ]:
query = (
    session.readStream
        .format("rate")
        .option("rowsPerSecond", 10)
        .option("numPartitions", 2)
        .load()
        .selectExpr("value", "value * 2 AS doubled")
        .where("value % 2 = 0")
        .writeStream
        .format("console")
        .trigger(continuous="1 second")
        .start()
)

query.awaitTermination(20)
query.stop()

### Processing 5 ⟶ Data stream REAL TIME processing

In [ ]:
# NOT RUNNABLE - KAFKA NEEDED
query = (
    session.readStream
        .format("rate")
        .option("rowsPerSecond", 10)
        .option("numPartitions", 2)
        .load()
        .selectExpr("value", "value * 2 AS doubled")
        .where("value % 2 = 0")
        .writeStream
        .format("console")
        .outputMode("update")
        .trigger(realTime="5 second")
        .start()
)
query.awaitTermination(20)
query.stop()

## Additions
1. Viewing list of databases;
2. Creation of the delta table;
3. Viewing version of the table in different moments of existence with usage of **time travel** mechanism.

In [ ]:
session.catalog.listDatabases()
weather.write.format("delta").saveAsTable("weather")
session.sql("DESCRIBE HISTORY weather").show()